In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Define paths
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data"
BREAD_PATH = DATA_PATH / "Bread"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)

MODELS_PATH = PROJECT_ROOT / "models"
MODELS_PATH.mkdir(exist_ok=True)

import pandas as pd
import numpy as np

from src.utils.splitting import split_last_n_observations
from src.models.autogluon_store import train_autogluon
from src.models.autogluon_product import build_ts_dataframe_product, train_autogluon_product
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from src.models.evaluate import plot_error_analysis
from src.utils.metrics import calculate_error_metrics

PREDICTION_LENGTH = 7  # horizon H
TEST_WINDOW_DAYS = 90  # must be >= H, gives ~13 evaluation weeks
RANDOM_SEED = 123

In [ ]:
# ======================
# Helper functions
# ======================
from autogluon.timeseries import TimeSeriesDataFrame


def make_future_known_covariates(
        hist_df,
        prediction_length,
        covariate_cols
):
    future_rows = []

    for item_id, g in hist_df.groupby("item_id"):
        last_date = g["timestamp"].max()
        future_dates = pd.date_range(
            start=last_date + pd.Timedelta(days=1),
            periods=prediction_length,
            freq="D")

        future_df = pd.DataFrame({
            "item_id": item_id,
            "timestamp": future_dates,
        })

        # calendar features (must be deterministic)
        future_df["day_of_week_num"] = future_df["timestamp"].dt.weekday
        future_df["month"] = future_df["timestamp"].dt.month
        future_df["quarter"] = future_df["timestamp"].dt.quarter

        future_df["sin_dow"] = np.sin(2 * np.pi * future_df["day_of_week_num"] / 7)
        future_df["cos_dow"] = np.cos(2 * np.pi * future_df["day_of_week_num"] / 7)
        future_df["sin_day"] = np.sin(2 * np.pi * future_df["timestamp"].dt.dayofyear / 365)
        future_df["cos_day"] = np.cos(2 * np.pi * future_df["timestamp"].dt.dayofyear / 365)

        # ⚠️ Promotion features: must be known or assumed
        for col in covariate_cols:
            if col not in future_df:
                future_df[col] = 0  # safe default if unknown

        future_rows.append(future_df)

    future_cov = pd.concat(future_rows)
    return TimeSeriesDataFrame.from_data_frame(
        future_cov,
        id_column="item_id",
        timestamp_column="timestamp",
    )


def total_wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

In [ ]:
# ---- Store-level data ----
store_df = pd.read_parquet(PROCESSED_PATH / "store_daily.parquet")
store_df["date"] = pd.to_datetime(store_df["date"])

store_train_df, store_test_df = split_last_n_observations(store_df, PREDICTION_LENGTH)

### AutoGluon baseline

#### Train STORE AutoGluon (without features)

In [ ]:
STORE_MODEL_NAME = "autogluon_store_baseline"

store_train_ag = store_train_df.rename(columns={
    "gln": "item_id",
    "date": "timestamp",
    "quantity": "target",
})

train_autogluon(
    train_df=store_train_ag,
    model_path=MODELS_PATH / STORE_MODEL_NAME,
    prediction_length=PREDICTION_LENGTH,
    use_features=False,  # True if we add store temporal features
)

#### Predict STORE test period

In [ ]:
store_predictor = TimeSeriesPredictor.load(
    MODELS_PATH / STORE_MODEL_NAME
)

store_test_ag = store_test_df.rename(columns={
    "gln": "item_id",
    "date": "timestamp",
    "quantity": "target",
})

In [ ]:
train_ts = TimeSeriesDataFrame.from_data_frame(
    store_train_ag,
    id_column="item_id",
    timestamp_column="timestamp",
)

test_ts = TimeSeriesDataFrame.from_data_frame(
    store_test_ag,
    id_column="item_id",
    timestamp_column="timestamp",
)

In [ ]:
store_predictions = store_predictor.predict(train_ts)

In [ ]:
store_pred_df = store_predictions.reset_index().rename(columns={"mean": "y_pred"})
store_truth_df = test_ts.reset_index().rename(columns={"target": "y_true"})
store_eval_df = store_truth_df.merge(store_pred_df, on=["item_id", "timestamp"], how="inner")

assert len(store_eval_df) > 0

In [ ]:
store_metrics = calculate_error_metrics(store_eval_df["y_true"].values, store_eval_df["y_pred"].values)

plot_error_analysis(store_eval_df["y_true"].values, store_eval_df["y_pred"].values)

In [ ]:
leaderboard_store = store_predictor.leaderboard(silent=True)
print(leaderboard_store[["model", "score_val"]])

In [ ]:
total_wape = np.abs(store_eval_df["y_true"] - store_eval_df["y_pred"]).sum() / np.abs(store_eval_df["y_true"]).sum()

print(f"STORE TOTAL WAPE: {total_wape:.3%}")

In [ ]:
store_eval_df["volume_bucket"] = pd.qcut(store_eval_df.groupby("item_id")["y_true"].transform("mean"), q=4,
                                         labels=["low", "mid-low", "mid-high", "high"])


def wape(df):
    return np.abs(df["y_true"] - df["y_pred"]).sum() / np.abs(df["y_true"]).sum()


bucket_wape = store_eval_df.groupby("volume_bucket", observed=True).apply(wape)

print(bucket_wape)

### AutoGluon + known covariates only

In [ ]:
# Note that we use the same dataset as baseline, 
# so we use store_train_ag, train_ts, and test_ts as defined for baseline model

STORE_MODEL_NAME_Feat = "autogluon_store_features_v1"

train_autogluon(
    train_df=store_train_ag,
    model_path=MODELS_PATH / STORE_MODEL_NAME_Feat,
    prediction_length=PREDICTION_LENGTH,
    use_features=True,  # True if we add store temporal features
)

In [ ]:
store_predictor_feat = TimeSeriesPredictor.load(
    MODELS_PATH / STORE_MODEL_NAME_Feat
)

In [ ]:
COVARIATE_COLS = store_predictor_feat.known_covariates_names

future_known_covariates = make_future_known_covariates(
    hist_df=store_train_ag,
    prediction_length=PREDICTION_LENGTH,
    covariate_cols=COVARIATE_COLS,
)

store_predictions_feat = store_predictor_feat.predict(train_ts, known_covariates=future_known_covariates)

In [ ]:
store_pred_df_feat = store_predictions_feat.reset_index().rename(columns={"mean": "y_pred"})

store_truth_df_feat = test_ts.reset_index().rename(columns={"target": "y_true"})

store_eval_df_feat = store_truth_df_feat.merge(store_pred_df_feat, on=["item_id", "timestamp"], how="inner")

In [ ]:
store_metrics_feat = calculate_error_metrics(store_eval_df_feat["y_true"].values, store_eval_df_feat["y_pred"].values)

plot_error_analysis(store_eval_df_feat["y_true"].values, store_eval_df_feat["y_pred"].values)

In [ ]:
leaderboard_store_feat = store_predictor_feat.leaderboard(silent=True)
print(leaderboard_store_feat[["model", "score_val"]])

In [ ]:
total_wape_feat = (np.abs(store_eval_df_feat["y_true"] - store_eval_df_feat["y_pred"]).sum()
                   / np.abs(store_eval_df_feat["y_true"]).sum())

print(f"STORE TOTAL WAPE: {total_wape_feat:.3%}")

In [ ]:
store_eval_df_feat["volume_bucket"] = pd.qcut(
    store_eval_df_feat.groupby("item_id")["y_true"]
    .transform("mean"),
    q=4,
    labels=["low", "mid-low", "mid-high", "high"],
    duplicates="drop",
)

bucket_wape_feat = (
    store_eval_df_feat
    .groupby("volume_bucket", observed=True)
    .apply(wape)
)

print(bucket_wape_feat)

### Chronos-only store model

In [ ]:
# Train Chronos-only store model
STORE_CHRONOS_NAME = "autogluon_store_chronos_only"

predictor_chronos = TimeSeriesPredictor(
    target="target",
    prediction_length=PREDICTION_LENGTH,
    freq="D",
    eval_metric="WAPE",
    path=MODELS_PATH / STORE_CHRONOS_NAME,
    verbosity=2,
)

predictor_chronos.fit(
    store_train_ag,
    presets="medium_quality",
    time_limit=3600,
    hyperparameters={
        "Chronos": {},
    },
)

In [ ]:
# Predict store baseline
store_predictions_chronos = predictor_chronos.predict(train_ts)

store_pred_df_chronos = (
    store_predictions_chronos
    .reset_index()
    .rename(columns={"mean": "y_pred"})
)

In [ ]:
# Evaluate
store_eval_df_chronos = (
    store_test_df
    .rename(columns={
        "gln": "item_id",
        "date": "timestamp",
        "quantity": "y_true",
    })
    .merge(
        store_pred_df_chronos,
        on=["item_id", "timestamp"],
        how="inner",
    )
)

store_metrics_chronos = calculate_error_metrics(
    store_eval_df_chronos["y_true"].values,
    store_eval_df_chronos["y_pred"].values,
)

store_metrics_chronos

In [ ]:
plot_error_analysis(store_eval_df_chronos["y_true"].values, store_eval_df_chronos["y_pred"].values, )

In [ ]:
leaderboard_store_chronos = predictor_chronos.leaderboard(silent=True)
print(leaderboard_store_chronos[["model", "score_val"]])

In [ ]:
total_wape_chronos = (
        np.abs(store_eval_df_chronos["y_true"] - store_eval_df_chronos["y_pred"]).sum()
        / np.abs(store_eval_df_chronos["y_true"]).sum()
)

print(f"STORE TOTAL WAPE: {total_wape_chronos:.3%}")

In [ ]:
store_eval_df_chronos["volume_bucket"] = pd.qcut(
    store_eval_df_chronos.groupby("item_id")["y_true"].transform("mean"),
    q=4,
    labels=["low", "mid-low", "mid-high", "high"],
)

bucket_wape_chronos = (
    store_eval_df_chronos
    .groupby("volume_bucket", observed=True)
    .apply(wape)
)

print(bucket_wape_chronos)

## Product model 

In [ ]:
# Model A — Pure Univariate (Baseline)
FEATURE_SET_A = []

# Model B — Full economic structure
FEATURE_SET_B = [
    "day_of_week_num",
    "on_promotion",
    "discount",
    "gb_id_mean_target",
    "gln_te",
    "price_relative",
]

# Model C — Simple calendar baseline
FEATURE_SET_C = [
    "day_of_week_num",
    "month",
    "is_holiday",
]

### Model A

In [ ]:
FEATURE_SET = FEATURE_SET_A  # change here only
PRODUCT_MODEL_NAME = f"autogluon_product_A_{len(FEATURE_SET)}"

prod_train_ts, used_features = build_ts_dataframe_product(prod_train_df, feature_cols=FEATURE_SET)

print("Using features:", used_features)

train_autogluon_product(
    train_ts=prod_train_ts,
    known_covariates=used_features,
    model_path=MODELS_PATH / PRODUCT_MODEL_NAME,
    prediction_length=PREDICTION_LENGTH,
    random_seed=RANDOM_SEED,
)

In [ ]:
# Load predictor
product_predictor = TimeSeriesPredictor.load(MODELS_PATH / PRODUCT_MODEL_NAME)

print("Known covariates:", product_predictor.known_covariates_names)

In [ ]:
PRODUCT_COVARIATES = product_predictor.known_covariates_names

product_future_covariates = make_future_known_covariates(
    hist_df=prod_train_df,
    prediction_length=PREDICTION_LENGTH,
    covariate_cols=PRODUCT_COVARIATES,
)

In [ ]:
# Predict
product_predictions = product_predictor.predict(prod_train_ts, known_covariates=product_future_covariates, )

product_pred_df = product_predictions.reset_index().rename(columns={"mean": "y_pred"})

print(product_pred_df.head())

### Model B

In [ ]:
FEATURE_SET = FEATURE_SET_B  # change here only
PRODUCT_MODEL_NAME = f"autogluon_product_B_{len(FEATURE_SET)}"

prod_train_ts, used_features = build_ts_dataframe_product(prod_train_df, feature_cols=FEATURE_SET)

print("Using features:", used_features)

train_autogluon_product(
    train_ts=prod_train_ts,
    known_covariates=used_features,
    model_path=MODELS_PATH / PRODUCT_MODEL_NAME,
    prediction_length=PREDICTION_LENGTH,
    random_seed=RANDOM_SEED,
)

In [ ]:
# Load predictor
product_predictor = TimeSeriesPredictor.load(MODELS_PATH / PRODUCT_MODEL_NAME)

print("Known covariates:", product_predictor.known_covariates_names)

In [ ]:
PRODUCT_COVARIATES = product_predictor.known_covariates_names

product_future_covariates = make_future_known_covariates(
    hist_df=prod_train_df,
    prediction_length=PREDICTION_LENGTH,
    covariate_cols=PRODUCT_COVARIATES,
)

In [ ]:
# Predict
product_predictions = product_predictor.predict(prod_train_ts, known_covariates=product_future_covariates)

product_pred_df = (
    product_predictions
    .reset_index()
    .rename(columns={"mean": "y_pred"})
)

print(product_pred_df.head())

### Model C

In [ ]:
FEATURE_SET = FEATURE_SET_C  # change here only
PRODUCT_MODEL_NAME = f"autogluon_product_C_{len(FEATURE_SET)}"

prod_train_ts, used_features = build_ts_dataframe_product(prod_train_df, feature_cols=FEATURE_SET)

print("Using features:", used_features)

train_autogluon_product(
    train_ts=prod_train_ts,
    known_covariates=used_features,
    model_path=MODELS_PATH / PRODUCT_MODEL_NAME,
    prediction_length=PREDICTION_LENGTH,
    random_seed=RANDOM_SEED,
)

In [ ]:
# Load predictor
product_predictor = TimeSeriesPredictor.load(MODELS_PATH / PRODUCT_MODEL_NAME)

print("Known covariates:", product_predictor.known_covariates_names)

In [ ]:
PRODUCT_COVARIATES = product_predictor.known_covariates_names

product_future_covariates = make_future_known_covariates(
    hist_df=prod_train_df,
    prediction_length=PREDICTION_LENGTH,
    covariate_cols=PRODUCT_COVARIATES,
)

In [ ]:
# Predict
product_predictions = product_predictor.predict(prod_train_ts, known_covariates=product_future_covariates)

product_pred_df = product_predictions.reset_index().rename(columns={"mean": "y_pred"})

print(product_pred_df.head())